# Harness对Agent运行的影响:3种典型的失败模式

## 失败模式一：目标丢失

假设你让Agent完成一个跨多个文件的重构任务。前几步，它表现正常：理解了任务、制定了计划、开始执行。随着会话轮数增加，早期的任务描述和约束条件逐渐被新的内容挤出上下文窗口

问题不在于模型的记忆力差，因为在同一个上下文窗口内，模型对信息的记忆是准确的。问题在于上下文管理机制的缺失。当窗口接近满载时，Harness应该主动压缩早期的历史会话，保留任务目标和关键约束，同时丢弃中间过程的冗余细节，保持Harness具备自动治理上下文能力

## 失败模式二：上下文混乱

Agent在处理大型项目时，每次读取文件、调用工具、接受报错信息。都会向上下文中注入新信息。如果没有有效的管理策略，上下文就会变成一个不断膨胀的“垃圾堆” ————大量已经用过的中间结果和重复信息挤占着有限的空间

需要以下两个机制

- 准入控制机制：并非所有的信息都需要完整地进入上下文
- 退出机制：对于已完成的步骤，将其详细执行记录压缩成一句摘要，以释放上下文空间

## 失败模式三：错误恢复失败

看到报错信息 -> 分析报错信息 -> 定位到方法名写错了 -> 回到源码查找正确的方法名 -> 修改测试代码 -> 重新运行测试 -> 成功


# Agent=Compose

Harness是指包裹在AI模型外面的一整套基础设施，涵盖六个方面

- 工具系统：Agent能够调用那些工具
- 权限控制：每个工具的使用是否需要授权，以及授权的粒度如何
- 上下文管理：如何组织系统提示词、如何控制上下文窗口、在窗口接近满载时如何压缩
- 反馈循环：Agent执行一步操作后能否自动验证结果、验证失败后能否自主修复
- 错误恢复：针对不同类型的错误，采取何种恢复策略
- 可观测性：Token消耗、执行耗时、行为日志的追踪与记录

> 提升Agent可靠性的最高效路径，往往不是更换更强的模型，而是改进Harness

| 阶段 | 核心问题 | 管理对象 | 典型产物 |
| --- | --- | --- | --- |
| Prompt Engineering | 怎么写指令 | 输入的指令 | 提示词模板、角色设定 |
| Context Engineering | 给模型看什么信息 | 上下文信息 | CLAUDE.md、文档架构 |
| Harness Engineering | 在什么运行环境中工作 | 运行环境与控制系统 | 权限系统、压缩策略、反馈循环 |

# Harness的六层参考架构

"Agent = Model + Harness"中的 Harness 部分可以工程化地拆成六层。每层对应一类失效模式，也对应一组可以独立测试、独立替换的组件。下面的代码示例来自开源项目 **harness-py**（一个千行左右的教学级编码 Agent，与 Claude Code 的设计同源），适合逐层对照阅读。

| 层 | 回答的问题 | 对应失败模式 | harness_py 对应模块 |
| --- | --- | --- | --- |
| 约束层 | 什么绝对不能做 | 不可逆破坏 | config.py / tools.py |
| 工具层 | 能做什么、怎么做 | 能力边界 | tools.py |
| 上下文层 | 每轮让模型看到什么 | 上下文混乱 | prompt.py |
| 记忆层 | 长任务如何记住关键信息 | 目标丢失 | token_budget.py / compressor.py / memory.py / session.py |
| 验证层 | 做错了如何及时发现 | 错误恢复失败 | loop_guard.py |
| 编排层 | 以上如何协同成一个运行时 | 全部 | agent.py |

## 约束层

**解决的问题**：模型生成的每个动作都可能产生不可逆副作用——删文件、覆盖代码、执行危险命令。写在 Prompt 里的"请不要删除文件"是软约束，可以被间接注入绕过；约束层要做的是把软约束变成代码强制执行的硬约束。

三道防线，从粗到细：

1. **能力开关**：启动时决定 Agent 拥有哪些能力，从源头收窄攻击面
2. **参数校验**：即使能力开放，每次调用前仍检查参数是否越界
3. **命令黑名单**：对 shell 命令做子串匹配，直接拒绝高危操作

```python
# config.py —— 第一道防线：能力开关（deny by default）
@dataclass
class AgentConfig:
    max_iterations: int = 200        # 迭代上限，防止无限运行
    max_cost_usd: float = 2.0        # 成本上限
    allow_write: bool = True         # 是否允许写文件
    allow_shell: bool = True         # 是否允许执行 shell
    allow_destructive: bool = False  # 危险操作默认关闭，需显式打开

    # 分阶段执行
    planning_turns: int = 3
    planning_tools: tuple = ('read_file', 'grep_search', 'glob_search')
```

```python
# tools.py —— 第二道防线：路径逃逸检查，防 ../../etc/passwd 式路径遍历
def _check_path_escape(config, raw_path) -> tuple[bool, str]:
    resolved = (Path(config.cwd) / raw_path).resolve()
    cwd_resolved = Path(config.cwd).resolve()
    # 用 relative_to 而不是 startswith，避免前缀碰撞（/project vs /project_secret）
    try:
        resolved.relative_to(cwd_resolved)
    except ValueError:
        return False, f'Path escapes working directory: {raw_path}'
    return True, ''

# tools.py —— 第三道防线：破坏性命令黑名单（子串匹配）
if not config.allow_destructive:
    dangerous = ['rm -rf', 'rmdir /s', 'format ', 'mkfs', 'dd if=', '> /dev/']
    for d in dangerous:
        if d in command:
            return False, f'Destructive command blocked: {d}'
```

三个设计要点：

- **默认拒绝**（deny by default）：`allow_destructive` 默认为 False，危险能力必须显式打开——安全工程中的最小权限原则
- **约束放在代码层而不是提示层**：Prompt 可能被注入覆盖，`_check_path_escape` 的返回值不受任何输入影响
- **一处反直觉的写法**：判断路径边界用 `relative_to` 而非 `startswith`，因为 `'/project'.startswith('/project')` 对 `'/project_secret'` 也会误判通过——安全代码要防住"看似等价"的写法

业界对照：Claude Code 的权限模式（default / acceptEdits / plan / bypassPermissions）本质上是同一组开关的不同预设；OpenAI Codex CLI 则把约束下沉到操作系统沙箱层。

## 工具层

**解决的问题**：模型只能通过工具与世界交互，工具集的设计直接决定 Agent 的能力上限——工具太少完不成任务，太多则选择错误率上升、Schema 膨胀挤占上下文。

关键机制：

1. **注册表 + Schema**：每个工具三件套——名字、JSON Schema 参数定义、可执行函数；注册表解耦"定义"与"执行"
2. **少而精**：6 个工具（读 / 写 / 改 / 搜索×2 / Shell）覆盖绝大部分编码任务
3. **分阶段暴露**：规划阶段只给只读工具，想清楚之前不允许动手
4. **严格失败**：`edit_file` 的 `old_string` 必须精确匹配，把"改错位置"这类静默错误变成显式报错

```python
# tools.py —— 注册表：名字 → 可执行函数
TOOL_REGISTRY = {
    'read_file': tool_read_file,
    'write_file': tool_write_file,
    'edit_file': tool_edit_file,
    'grep_search': tool_grep_search,
    'glob_search': tool_glob_search,
    'bash': tool_bash,
}

# tools.py —— 分阶段工具暴露
def get_schemas_for_phase(turn, config) -> list[dict]:
    """前 planning_turns 轮只暴露只读工具，之后解锁全部。"""
    if turn <= config.planning_turns:
        allowed = set(config.planning_tools)
        return [s for s in TOOL_SCHEMAS if s['name'] in allowed]
    return list(TOOL_SCHEMAS)

# 越权调用不会静默执行，而是返回可读错误让模型自我纠正
if turn <= config.planning_turns and name not in config.planning_tools:
    return False, f'规划阶段（前{config.planning_turns}轮）不允许{name}。请先输出修改计划。'
```

```python
# tools.py —— 严格失败示例：edit_file（节选）
old = args['old_string']
if old not in text:
    return False, f'String not found in {args["path"]}'
count = text.count(old)
text = text.replace(old, args['new_string'], 1)
return True, f'Replaced 1/{count} occurrence(s) in {path.name}'
# count > 1 时也只替换第一处并如实报告——宁可保守，不可静默错改
```

业界经验：

- Anthropic《Writing effective tools for agents》：工具的 description 是最被低估的提示工程位置；错误信息要**可操作**（告诉模型下一步怎么办），而不是抛裸异常
- OpenAI 的实践：让 Linter 报错自带修复建议，Agent 拿到反馈一轮就能修好
- MCP（Model Context Protocol）正在把"工具注册 + Schema + 发现"标准化，工具层的思路已成为行业协议

## 上下文层

**解决的问题**：失败模式二"上下文混乱"。上下文是 Agent 唯一的感知通道，但不是所有信息都值得完整进入窗口——这一层负责**准入控制**：每轮该让模型看到什么、以什么形式、多大体量看到。

harness_py 在 `build_system_prompt` 中实现了四个机制：

1. **静态 / 动态分离**：系统提示词中不变的部分与每轮变化的部分（日期、工作目录、注入的额外上下文）分开，中间插入 Cache 边界标记，让 Prompt Cache 命中静态部分
2. **项目记忆自动发现**：按 Claude Code 的规则查找 CLAUDE.md / AGENTS.md / .claude/rules/
3. **注入威胁扫描**：每个待注入的上下文文件先过一遍危险模式正则，命中即整体屏蔽
4. **体量裁剪**：单个上下文文件 40K 字符上限（对齐 Claude Code），工具输出按 `max_output_chars` 截断

```python
# prompt.py —— 上下文文件发现：项目级 → 父目录 → 全局 → rules 目录
def discover_claude_md(cwd: Path) -> list[tuple[Path, str]]:
    candidates = [cwd / 'CLAUDE.md', cwd / '.claude' / 'CLAUDE.md',
                  cwd / 'CLAUDE.local.md', cwd / 'AGENTS.md']
    for parent in cwd.resolve().parents:              # 向上遍历父目录
        candidates.append(parent / 'CLAUDE.md')
    candidates.append(Path.home() / '.claude' / 'CLAUDE.md')   # 全局
    rules_dir = cwd / '.claude' / 'rules'
    if rules_dir.is_dir():
        candidates.extend(sorted(rules_dir.glob('*.md')))
    ...
```

```python
# prompt.py —— Hermes 风格注入扫描：命中即屏蔽整个文件（节选 6 类，源码共 8 类）
_THREAT_PATTERNS = [
    re.compile(r'ignore\s+(all\s+)?(previous|above|prior)\s+(instructions|prompts)', re.I),
    re.compile(r'you\s+are\s+now\s+(a|an)\s+', re.I),                       # 角色覆写
    re.compile(r'cat\s+[~./]*\.env', re.I),                                  # 诱导读密钥
    re.compile(r'curl\s+.*\|\s*sh', re.I),                                   # 诱导执行远程脚本
    re.compile(r'<\s*div\s+style\s*=\s*["\'].*display\s*:\s*none', re.I),  # 隐形文本
    re.compile(r'[\u200b\u200c\u200d\ufeff]{3,}'),                           # 隐藏Unicode序列
]

threats = scan_context_threats(content)
if threats:
    content = f'[BLOCKED: {resolved.name} contains suspicious patterns: ...]'
```

```python
# prompt.py —— Cache 边界：静态与动态分离
parts.append('You are Harness-py, ...')            # 静态：角色、执行协议（可缓存）
parts.append('## Project Context')                  # 静态：CLAUDE.md 内容（可缓存）
parts.append(f'\n{_DYNAMIC_BOUNDARY}')              # ← Cache 边界标记
parts.append(f'Current date: {datetime.date.today().isoformat()}')  # 动态：每轮变化
parts.append(f'Working directory: {cwd}')
```

值得记住的经验数字与原则：

- 上下文利用率超过约 **40%** 后，长任务成功率开始明显下滑（Anthropic 称之为 Dumb Zone）——上下文层要主动维持低占用，而不是等满了再处理
- 准入控制的首选不是"预先塞满"，而是**按需加载**（Just-in-Time Context）：需要时再用工具去查，工具调用本身就是一种上下文供给

## 记忆层

**解决的问题**：失败模式一"目标丢失"。窗口有限而任务时间线很长，记忆层负责**退出机制**：让已完成步骤的细节离开窗口，让关键事实跨轮次乃至跨会话存活。

harness_py 把记忆拆成四个子系统：

| 子系统 | 模块 | 职责 |
| --- | --- | --- |
| Token 预算 | token_budget.py | 把窗口划分为五个区，压缩时有明确目标 |
| 四级压缩 | compressor.py | 按代价递进：能用便宜手段就不用贵的 |
| 跨会话记忆 | memory.py | 长期事实落盘，下次会话围栏注入 |
| 会话持久化 | session.py | 每事件落盘 jsonl，崩溃可恢复、可续跑 |

```python
# token_budget.py —— 五区预算：把"还没发生"的部分预留出来
@classmethod
def allocate(cls, context_window: int) -> TokenBudget:
    output  = int(context_window * 0.15)   # 输出预留
    system  = int(context_window * 0.10)   # 系统提示词
    memory  = int(context_window * 0.05)   # 注入的长期记忆
    current = int(context_window * 0.20)   # 当前任务描述
    history = context_window - output - system - memory - current  # 会话历史 ≈ 50%

# 判断是否压缩：先看是否侵占输出预留（紧急），再看是否越过 80% 阈值
def should_compress(budget, total_tokens, threshold_pct=0.80) -> tuple[bool, str]:
    if total_tokens > budget.context_window - budget.output_reserve:
        return True, f'紧急：侵占了输出预留（{total_tokens:,} > ...）'
    if total_tokens > threshold:
        return True, f'超过压缩阈值（{threshold_pct:.0%}）'
```

```python
# compressor.py —— 四级压缩（由轻到重，逐级升级）
def compress(self, messages, target_tokens, llm_call):
    msgs = self._microcompact(messages, preserve)    # L1 截断超长工具输出，保留首尾
    if ...:
        msgs = self._snip(msgs, preserve)            # L2 已完成步骤的完整结果 → 一行预览
    if ...:
        msgs = self._compact(msgs, preserve, llm_call)  # L3 LLM 迭代摘要
    return self._fix_orphaned_tool_pairs(msgs)       # L4 补齐孤立 tool 消息，防 API 报错
```

memory.py 与 session.py 补齐"跨会话"维度：

- **按项目隔离**：记忆写入 `~/.claude/projects/{项目路径key}/memory/`，不同项目互不污染
- **Hermes 围栏注入**：恢复的记忆包在 `<memory-context>` 标签内，注明"以下是历史记忆，以当前用户消息为准"——防止旧记忆被当作新指令
- **dream() 定期整理**：模仿睡眠整理记忆，四阶段处理（合并、删除、日期修正、索引重建）
- **事件溯源**：session.py 把每条消息作为 JSON 事件立即写盘（对齐 Claude Code 的 jsonl 格式），恢复时重放事件流重建对话；写一半崩溃最多丢最后一条

业界对照——Anthropic《Effective harnesses for long-running agents》的组合拳：把任务拆成 feature list（JSON）逐项打勾；每完成一项就 git commit；**进度写在文件里而不是对话里**——即使上下文被压缩甚至换新会话，进度也不丢。这与 session.py / memory.py 是同一原理的两个侧面：把状态外置到窗口之外。

## 验证层

**解决的问题**：失败模式三的恢复循环里，第一步是"看到报错"。LLM 会自信地犯错且不会主动承认，验证层要在模型之外建立客观反馈回路：发现问题 → 判定严重程度 → 决定继续、提醒还是终止。

harness_py 的验证分两个时刻：

- **执行前**：约束层的参数校验保证错误输入不进入执行
- **执行后**：每个工具返回统一的 `(ok, content)`，LoopGuard 在每次调用后检查行为是否异常

```python
# loop_guard.py —— 四种失控检测（核心逻辑节选）
@dataclass
class LoopGuard:
    max_identical_calls: int = 3      # 相同调用连续出现次数
    max_consecutive_errors: int = 5   # 连续失败次数
    max_same_tool_streak: int = 10    # 同一工具连击次数
    max_total_tool_calls: int = 100   # 单次运行总调用上限

    def check(self, tool_name, tool_args, result_ok, result_preview) -> tuple[bool, str]:
        # 返回 (should_intervene, message)
        self._total_calls += 1
        if self._total_calls > self.max_total_tool_calls:
            return True, f'工具调用总量已达{self._total_calls}次。请总结进度并停止。'
        if not result_ok:
            self._consecutive_errors += 1
            if self._consecutive_errors >= self.max_consecutive_errors:
                return True, f'已连续{self._consecutive_errors}次失败。请换一种方法或报告无法完成。'
        else:
            self._consecutive_errors = 0
        # ...同工具连击检测（max_same_tool_streak）...
        # 调用签名 = 工具名 + 参数md5 + 结果预览，识别"换汤不换药"的重试
        args_hash = hashlib.md5(json.dumps(tool_args, sort_keys=True, default=str).encode()).hexdigest()[:12]
        sig = f'{tool_name}:{args_hash}:{result_preview[:200]}'
        # ...最近 max_identical_calls 次签名完全相同 → 死循环...
        if len(set(tail)) == 1:
            return True, (f'检测到死循环：最近{self.max_identical_calls}次调用完全相同（{tool_name}）。'
                          f'请立即停止重试，换一种完全不同的策略。')
        return False, ''
```

干预方式值得注意：Guard 触发后**不是直接终止进程**，而是把警告作为一条 user 消息注入对话——把"人喊停"变成"Harness 递纸条"，让模型自己修正：

```python
# agent.py —— Guard 干预 = 注入系统提醒，而非硬终止
intervene, guard_msg = guard.check(tool_name, tool_args, ok, tool_content[:200])
if intervene:
    messages.append({'role': 'user',
                     'content': f'<system-reminder>[LOOP GUARD] {guard_msg}</system-reminder>'})
```

四种检测对应的真实故障：

| 检测 | 对应故障 |
| --- | --- |
| 相同调用重复 | 模型用同样的错误参数反复重试 |
| 连续错误 | 环境坏了（如依赖缺失），重试无意义 |
| 同工具连击 | 陷入偏执行为，如无限读文件 |
| 总量上限 | 最坏情况的兜底闸门 |

业界对照：

- Anthropic 的 initializer + coding agent 分离：initializer 搭好脚手架，coding agent 在干净上下文中工作，避免调试历史污染主任务
- Planner / Generator / Evaluator 三角色架构：验证逻辑独立于生成逻辑，防止"自己检查自己"的宽容偏差
- 对编码 Agent，最可靠的验证器是测试套件：执行 pytest 本身就是最诚实的反馈回路

## 编排层

**解决的问题**：前五层是五个零件，编排层把它们装配成一个运行时：决定每轮做什么、何时压缩、何时换工具集、何时停止、出错怎么办。

```python
# agent.py —— run() 主循环（简化注释版）
for iteration in range(1, ac.max_iterations + 1):
    # ① 预检压缩：调 API 之前先算 token，接近阈值就先压缩
    current_tokens = compressor.total_tokens(messages)
    need, reason = should_compress(budget, current_tokens, ac.compress_threshold_pct)
    if need:
        messages = compressor.compress(messages, threshold, llm_call=llm_summarize)

    # ② 分阶段工具解锁：规划阶段只读，之后全量
    phase_schemas = get_schemas_for_phase(iteration, ac)

    # ③ 调用 LLM；报错分级处理：上下文类错误 → 紧急压缩后重试
    try:
        response = client.complete(messages, phase_schemas)
    except RuntimeError as exc:
        if 'context' in str(exc).lower() or 'token' in str(exc).lower():
            messages = compressor.compress(messages, int(mc.context_window * 0.5), reactive=True)
            response = client.complete(messages, phase_schemas)   # 最后防线
        else:
            break

    # ④ 自然停止：模型不再发起工具调用 = 认为任务完成
    if not tool_calls:
        result.stop_reason = 'stop'
        break

    # ⑤ 逐个执行工具 + Guard 检查，全程写 session
    for tc in tool_calls:
        ok, tool_content = execute_tool(tool_name, tool_args, ac, turn=iteration)
        intervene, guard_msg = guard.check(tool_name, tool_args, ok, tool_content[:200])
        if intervene:
            messages.append({'role': 'user',
                             'content': f'<system-reminder>[LOOP GUARD] {guard_msg}</system-reminder>'})
```

四个编排决策：

1. **压缩时机前置**：在调用 API 前预检（阈值 80%），而不是等 API 报 `context_length_exceeded` 再补救；API 报错路径上还保留一次"紧急压缩重试"——两层防御
2. **自然停止**：不硬编码"任务完成判断"，把控制权交给模型——它不再发起工具调用时视为完成；`max_iterations` 只是兜底
3. **故障自愈**：上下文类 API 错误压缩后重试，其他错误如实终止并报告原因（stop_reason）
4. **可恢复运行**：`resume(session_id)` 从 jsonl 重放历史消息，并刷新 system prompt（CLAUDE.md 和日期可能已变化）——任务跨进程存活

六层协同全景（一次工具调用的完整路径）：

![Harness 六层协同运行时图](../src/harness-six-layer-runtime.svg)

最后留一个自查问题：当你手上的 Agent 出错时，能否定位到是哪一层的缺失？目标漂移是记忆层问题，重复失败是验证层问题，误删文件是约束层问题。六层架构提供的不是标准答案，而是一张**故障定位地图**——这也是 Harness Engineering 与 Prompt Engineering 最大的区别：前者让你可以分层归因、分层改进。